In [1]:
import os
fig_dir = "/home/gpu_data/data8/kinoshita_simulation/synapse_sleep_Fig/"
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
#mean field representative ones,  different U, w  phase plane, vector map, time change in FR  -> simu fig
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Oscillation map (mean-field only) + (r,f) phase plane:
- Model: 3-variable mean-field (s, r, f) with mu(t) = E_L + I0 + w*K*s(t)
- Sweep U (vertical axis) and w (horizontal axis) -> optional
- Color: yellow gradient encodes oscillation frequency [Hz]
- Black: non-oscillatory
- NEW: r-nullcline (analytic) + f-nullcline (df/dt=0 contour) + streamplot on (r,f) plane
       By default, eliminate s adiabatically: s = tau_s * U * r * f (fast synapse).
       If you prefer, set s_mode="fixed" to use an empirical s* estimated from MF time series.

Requirements:
  pip install numpy matplotlib scipy
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from math import sqrt, pi
from scipy.special import erfcx
# from scipy.optimize import brentq  # optional if you later want to solve for fixed points exactly

# -----------------------------
# Global parameters (no J_eff)
# -----------------------------
P = dict(
    tau_rec=0.789,   # s
    tau_f=0.02,      # s
    E_L=-65.0,       # mV
    V_th=-45.0,      # mV
    V_r=-65.0,       # mV
    tau_m=0.02,      # s
    tau_ref=0.002,   # s
    I0=10.63,        # mV
    sigma0=5.13      # mV
)

# Network constants that enter mean-field via mu = E_L + I0 + w*K*s
K = 30
TAU_S = 0.003  # s (fixed, as requested)

# Integration settings
Tsim   = 15.0     # total simulated seconds
dt_mf  = 1e-3     # 1 ms time step (fine enough for peak detection)

# Oscillation detection settings (as per the user's reference function)
LAST_SECONDS = 10.0
MIN_PEAKS    = 6
AMP_THR      = 1.5  # Hz

# Sweep ranges (adjust as needed)
U_min, U_max, nU = 0.01, 1.0, 60
w_min, w_max, nW = 30.0,  250.0, 60



# ===========================
# Utilities
# ===========================
def build_FI_mu_lut(p: dict, mu_min=-90.0, mu_max=-20.0, n_mu=1201, n_int=60):
    """Siegert transfer F_I(mu) lookup table for fixed noise sigma0 and LIF params."""
    def rate(mu):
        sigma = float(max(p["sigma0"], 1e-9))
        a = (p["V_r"]  - mu) / sigma
        b = (p["V_th"] - mu) / sigma
        us = np.linspace(a, b, n_int, dtype=float)
        I = np.trapz(erfcx(-us), us)
        T = p["tau_ref"] + p["tau_m"] * sqrt(pi) * I
        return 0.0 if T <= 1e-12 else max(0.0, 1.0 / T)

    mu_grid = np.linspace(mu_min, mu_max, n_mu, dtype=float)
    FI_grid = np.array([rate(mu) for mu in mu_grid], dtype=float)
    return mu_grid, FI_grid


def simulate_mf_3var_w(p: dict, U: float, w: float, tau_s: float, K: int,
                       mu_grid, FI_grid, Tsim: float, dt: float):
    """Mean-field 3-var RK4 with mu = E_L + I0 + w*K*s."""
    def FI_interp(mu):
        mu = np.clip(mu, mu_grid[0], mu_grid[-1])
        return np.interp(mu, mu_grid, FI_grid)

    n = int(round(Tsim / dt))
    t = np.linspace(0.0, n * dt, n + 1)

    s = np.empty(n + 1); r = np.empty(n + 1); f = np.empty(n + 1)
    s[0] = 0.0; r[0] = 1.0; f[0] = 10.0  # Hz

    wK = float(w) * float(K)

    def rhs(sv, rv, fv):
        sv = float(np.clip(sv, 0.0, 30.0))
        rv = float(np.clip(rv, 0.0, 1.2))
        fv = float(np.clip(fv, 0.0, 800.0))

        mu = p["E_L"] + p["I0"] + wK * sv
        ds = (-sv / tau_s) + U * rv * fv
        dr = (1.0 - rv) / p["tau_rec"] - U * rv * fv
        df = (-fv + FI_interp(mu)) / p["tau_f"]
        return ds, dr, df

    for k in range(n):
        sk, rk, fk = s[k], r[k], f[k]
        k1s, k1r, k1f = rhs(sk, rk, fk)
        k2s, k2r, k2f = rhs(sk + 0.5 * dt * k1s, rk + 0.5 * dt * k1r, fk + 0.5 * dt * k1f)
        k3s, k3r, k3f = rhs(sk + 0.5 * dt * k2s, rk + 0.5 * dt * k2r, fk + 0.5 * dt * k2f)
        k4s, k4r, k4f = rhs(sk + dt * k3s, rk + dt * k3r, fk + dt * k3f)

        s[k + 1] = np.clip(sk + (dt / 6.0) * (k1s + 2*k2s + 2*k3s + k4s), 0.0, 30.0)
        r[k + 1] = np.clip(rk + (dt / 6.0) * (k1r + 2*k2r + 2*k3r + k4r), 0.0, 1.2)
        f[k + 1] = np.clip(fk + (dt / 6.0) * (k1f + 2*k2f + 2*k3f + k4f), 0.0, 800.0)

    return t, s, r, f


# The user's reference routine (kept as-is, minor formatting only)
def per_cycle_amplitude(t, f, last_seconds: float = 8.0,
                        min_peaks: int = 6, amp_thr: float = 1.5):
    """
    Returns (amp_median, is_oscillatory, npeaks).
    amp_median is the median of (max-min) per cycle over the last window.
    """
    t0 = max(0.0, t[-1] - last_seconds)
    mask = t >= t0
    tt = t[mask]
    ff = f[mask]
    if ff.size < 200:
        return np.nan, False, 0

    d = np.diff(ff)
    peak_idx = np.where((d[:-1] > 0) & (d[1:] <= 0))[0] + 1
    if peak_idx.size < 4:
        return np.nan, False, peak_idx.size

    amp_global = float(ff.max() - ff.min())
    thr = float(ff.min() + 0.3 * amp_global)
    peak_idx = peak_idx[ff[peak_idx] >= thr]
    if peak_idx.size < 4:
        return np.nan, False, peak_idx.size

    amps = []
    for k in range(peak_idx.size - 1):
        i0, i1 = peak_idx[k], peak_idx[k+1]
        if i1 <= i0 + 3:
            continue
        seg = ff[i0:i1+1]
        amps.append(float(seg.max() - seg.min()))

    if len(amps) < 3:
        return np.nan, False, peak_idx.size

    amp_med = float(np.median(amps))
    is_osc = (peak_idx.size >= min_peaks) and (amp_global >= amp_thr) and (amp_med >= 0.5*amp_thr)
    return amp_med, is_osc, peak_idx.size


def estimate_frequency_hz(t, f, last_seconds: float = 8.0):
    """Median peak-to-peak frequency [Hz] over the last window; NaN if unreliable."""
    t0 = max(0.0, t[-1] - last_seconds)
    mask = t >= t0
    tt = t[mask]
    ff = f[mask]
    if ff.size < 200:
        return np.nan

    d = np.diff(ff)
    peak_idx = np.where((d[:-1] > 0) & (d[1:] <= 0))[0] + 1
    if peak_idx.size < 4:
        return np.nan

    amp_global = float(ff.max() - ff.min())
    thr = float(ff.min() + 0.3 * amp_global)
    peak_idx = peak_idx[ff[peak_idx] >= thr]
    if peak_idx.size < 2:
        return np.nan

    periods = np.diff(tt[peak_idx])
    periods = periods[periods > 1e-6]
    if periods.size == 0:
        return np.nan
    return float(np.median(1.0 / periods))


# ===========================
# NEW: (r,f) phase plane with nullclines + streamplot
# ===========================
def draw_phase_plane_r_f(P: dict, U: float, w: float, tau_s: float, K: int,
                         mu_grid, FI_grid,
                         s_mode: str = "adiabatic",   # "adiabatic" or "fixed"
                         s_fixed = "fixed", # if s_mode="fixed" and None -> estimate from MF sim
                         r_lim=(0.0, 1.1),
                         f_lim=(0.0, 120.0),
                         n_grid: int = 220,
                         density: float = 1.2,
                         figsize=(7.0, 5.5),
                         estimate_T: float = 4.0):
    """
    Draw (r,f) phase plane:
      - r-nullcline (analytic): dr/dt = 0 -> r = 1 / (1 + tau_rec * U * f)
      - f-nullcline (contour): df/dt = 0
      - Streamlines of the reduced 2D field (r,f)
    s handling:
      - "adiabatic": s = tau_s * U * r * f (s-nullcline; fast synapse)
      - "fixed":     s = s_fixed (if None, estimate from a short MF simulation and use the last 1 s average)
    """
    # Interpolant
    def FI_interp(mu):
        mu = np.clip(mu, mu_grid[0], mu_grid[-1])
        return np.interp(mu, mu_grid, FI_grid)

    # Resolve s* if needed
    if s_mode == "fixed" and (s_fixed is None):
        # brief run to estimate s* (mean over the last 1 s)
        t_tmp, s_tmp, _, _ = simulate_mf_3var_w(P, U, w, tau_s, K, mu_grid, FI_grid,
                                                Tsim=max(estimate_T, 2.0), dt=dt_mf)
        mask_last = t_tmp >= (t_tmp[-1] - 1.0)
        s_fixed = float(np.mean(s_tmp[mask_last]))
        print(f"[phase-plane] Estimated s* = {s_fixed:.4f} (fixed mode)")

    # Grid
    r_min, r_max = r_lim
    f_min, f_max = f_lim
    r = np.linspace(r_min, r_max, n_grid)
    f = np.linspace(f_min, f_max, n_grid)
    R, F = np.meshgrid(r, f)  # shapes: (ny, nx)

    # s on grid
    if s_mode == "adiabatic":
        S = tau_s * U * R * F
        s_label = r"$s=\tau_s U r f$ (adiabatic)"
    elif s_mode == "fixed":
        S = np.full_like(R, float(s_fixed))
        s_label = fr"$s={s_fixed:.4f}$ (fixed)"
    else:
        raise ValueError("s_mode must be 'adiabatic' or 'fixed'.")

    # Vector field (r', f')
    mu = P["E_L"] + P["I0"] + (w * K) * S
    FI_mu = FI_interp(mu)

    dR = (1.0 - R) / P["tau_rec"] - U * R * F
    dF = (-F + FI_mu) / P["tau_f"]

    # Normalize for nicer streamlines (avoid huge speeds)
    speed = np.hypot(dR, dF)
    scale = np.percentile(speed, 90)
    scale = scale if np.isfinite(scale) and scale > 0 else 1.0
    dRn = dR / scale
    dFn = dF / scale

    # r-nullcline (analytic)
    f_line = np.linspace(f_min, f_max, 1000)
    r_nc = 1.0 / (1.0 + P["tau_rec"] * U * f_line)
    # clip to display range
    r_nc = np.clip(r_nc, r_min, r_max)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)

    # Streamplot
    sp = ax.streamplot(r, f, dRn, dFn, color="0.6", linewidth=1.0, density=density, arrowsize=1.2)

    # r-nullcline
    ax.plot(r_nc, f_line, color="tab:orange", lw=2.5, label="r-nullcline")

    # f-nullcline (df/dt = 0)
    CS = ax.contour(R, F, dF, levels=[0.0], colors=["tab:blue"], linewidths=2.5)
    CS.collections[0].set_label("f-nullcline")

    # 固定点解析を実行し、図に重畳
    fps = classify_fixed_points_r_f(P, U, w, tau_s, K, mu_grid, FI_grid,
                                    s_mode=s_mode, s_fixed=None if s_mode=="adiabatic" else s_fixed,
                                    r_lim=r_lim, f_lim=f_lim,
                                    grid_coarse=40, verbose=True)

    plot_fixed_points_on_phase_plane(ax, fps, marker_sizes=(80, 70))

    ax.set_xlim(r_min, r_max)
    ax.set_ylim(f_min, f_max)
    ax.set_xlabel("r")
    ax.set_ylabel("f (Hz)")
#     ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(fig_dir + "/mean-field_streamplot_K{}_U{}_w{}.SVG".format(K, U, w))
    plt.show()
    
    
#     fig, ax = plt.subplots(figsize=figsize)

#     # Streamplot
#     sp = ax.streamplot(r, f, dRn, dFn, color="0.6", linewidth=1.0, density=density, arrowsize=1.2)

#     # r-nullcline
#     ax.plot(r_nc, f_line, color="tab:orange", lw=2.5)

#     # f-nullcline (df/dt = 0)
#     CS = ax.contour(R, F, dF, levels=[0.0], colors=["tab:blue"], linewidths=2.5)
# #     CS.collections[0].set_label("f-nullcline")

#     ax.set_xlim(r_min, r_max)
#     ax.set_ylim(f_min, f_max)
#     ax.set_xlabel("r")
#     ax.set_ylabel("f (Hz)")
# #     ax.set_title(f"(r,f) phase plane @ U={U:.3f}, w={w:.1f} mV, K={K}\n{ s_label }")
# #     ax.grid(True, alpha=0.25)
# #     ax.legend(loc="best")
#     plt.tight_layout()
# #     plt.savefig(fig_dir + "/mean-field_streamplot_K{}_U{}_w{}.SVG".format(K, U, w))
#     plt.show()
    
    
from scipy.optimize import root

def classify_fixed_points_r_f(P: dict, U: float, w: float, tau_s: float, K: int,
                              mu_grid, FI_grid,
                              s_mode: str = "adiabatic",
                              s_fixed ="fixed",
                              r_lim=(0.0, 1.1),
                              f_lim=(0.0, 150.0),
                              grid_coarse: int = 40,
                              tol_root: float = 1e-9,
                              tol_merge_r: float = 1e-3,
                              tol_merge_f: float = 1e-2,
                              verbose: bool = True):
    """
    固定点 (r*, f*) を探索して安定性を判定。
    戻り値: list of dicts with keys:
        r, f, kind, eigvals (complex np.array), J (2x2), s_star
    kind は 'stable focus', 'unstable focus', 'stable node', 'unstable node', 'saddle', 'degenerate' のいずれか。
    """

    # FI 補間
    def FI_interp(mu):
        mu = np.clip(mu, mu_grid[0], mu_grid[-1])
        return np.interp(mu, mu_grid, FI_grid)

    # s* の扱い
    if s_mode == "fixed" and (s_fixed is None):
        t_tmp, s_tmp, _, _ = simulate_mf_3var_w(P, U, w, tau_s, K, mu_grid, FI_grid, Tsim=4.0, dt=dt_mf)
        mask_last = t_tmp >= (t_tmp[-1] - 1.0)
        s_fixed = float(np.mean(s_tmp[mask_last]))

    def s_of(r, f):
        if s_mode == "adiabatic":
            return tau_s * U * r * f
        else:
            return float(s_fixed)

    # 連立方程式 g=0, h=0
    def g(r, f):
        return (1.0 - r)/P["tau_rec"] - U * r * f

    def h(r, f):
        mu = P["E_L"] + P["I0"] + (w * K) * s_of(r, f)
        return (-f + FI_interp(mu)) / P["tau_f"]

    # 数値微分（中心差分）
    def jacobian(r, f, eps_r=1e-6, eps_f=1e-4):
        gr = g(r, f); hr = h(r, f)
        g_rp = g(r + eps_r, f); g_rm = g(r - eps_r, f)
        g_fp = g(r, f + eps_f); g_fm = g(r, f - eps_f)
        h_rp = h(r + eps_r, f); h_rm = h(r - eps_r, f)
        h_fp = h(r, f + eps_f); h_fm = h(r, f - eps_f)
        dgdr = (g_rp - g_rm) / (2*eps_r)
        dgdf = (g_fp - g_fm) / (2*eps_f)
        dhdr = (h_rp - h_rm) / (2*eps_r)
        dhdf = (h_fp - h_fm) / (2*eps_f)
        return np.array([[dgdr, dgdf],[dhdr, dhdf]], dtype=float)

    # 粗探索: グリッドで符号変化ボックスを検出
    r_vals = np.linspace(r_lim[0], r_lim[1], grid_coarse)
    f_vals = np.linspace(f_lim[0], f_lim[1], grid_coarse)

    seeds = []
    G = np.empty((grid_coarse, grid_coarse))
    H = np.empty((grid_coarse, grid_coarse))
    for i, rf in enumerate(r_vals):
        for j, ff in enumerate(f_vals):
            G[i, j] = g(rf, ff)
            H[i, j] = h(rf, ff)

    for i in range(grid_coarse - 1):
        for j in range(grid_coarse - 1):
            # 四隅で g と h の符号がそれぞれ変化していれば近傍に根がある可能性
            g_block = [G[i,j], G[i+1,j], G[i,j+1], G[i+1,j+1]]
            h_block = [H[i,j], H[i+1,j], H[i,j+1], H[i+1,j+1]]
            if (np.min(g_block) <= 0 <= np.max(g_block)) and (np.min(h_block) <= 0 <= np.max(h_block)):
                rc = 0.5*(r_vals[i] + r_vals[i+1])
                fc = 0.5*(f_vals[j] + f_vals[j+1])
                seeds.append((rc, fc))

    # ニュートン法で精密化
    roots = []
    for (r0, f0) in seeds:
        def F(x):
            return np.array([g(x[0], x[1]), h(x[0], x[1])], dtype=float)

        sol = root(F, x0=np.array([r0, f0], dtype=float), jac=None, tol=tol_root, method='hybr')
        if sol.success:
            r_star, f_star = float(sol.x[0]), float(sol.x[1])
            # 範囲内・実数・妥当性チェック
            if (r_lim[0]-1e-6 <= r_star <= r_lim[1]+1e-6) and (f_lim[0]-1e-3 <= f_star <= f_lim[1]+1e-3):
                roots.append((r_star, f_star))

    # 重複マージ
    uniq = []
    for r_star, f_star in roots:
        found = False
        for k, (rr, ff) in enumerate(uniq):
            if abs(r_star - rr) < tol_merge_r and abs(f_star - ff) < tol_merge_f:
                found = True
                break
        if not found:
            uniq.append((r_star, f_star))

    results = []
    for r_star, f_star in uniq:
        J = jacobian(r_star, f_star)
        eig = np.linalg.eigvals(J)
        # 分類
        re = np.real(eig)
        im = np.imag(eig)
        if np.allclose(im, 0, atol=1e-9):  # 実固有値
            if re[0] * re[1] < 0:
                kind = "saddle"
            else:
                if np.all(re < 0):
                    kind = "stable node"
                elif np.all(re > 0):
                    kind = "unstable node"
                else:
                    kind = "degenerate"
        else:  # 複素共役
            if np.all(re < 0):
                kind = "stable focus"
            elif np.all(re > 0):
                kind = "unstable focus"
            else:
                # 一方が正・負の実部を持つ複素対はないはずだが安全側
                kind = "saddle/focus-mix"

        results.append(dict(
            r=r_star, f=f_star, kind=kind, eigvals=eig, J=J, s_star=s_of(r_star, f_star)
        ))

    if verbose:
        print("=== Fixed point analysis (r,f) ===")
        print(f"U={U:.4f}, w={w:.4f} mV, K={K}, s_mode={s_mode}" + (f", s_fixed={s_fixed:.6f}" if s_mode=='fixed' else ""))
        if len(results) == 0:
            print("No fixed points found in the specified window.")
        for i, fp in enumerate(results, 1):
            e1, e2 = fp["eigvals"]
            print(f"[{i}] r*={fp['r']:.6f}, f*={fp['f']:.6f}, s*={fp['s_star']:.6f}  -> {fp['kind']}")
            print(f"     eigvals: {e1:.6e}, {e2:.6e}")
            print(f"     J = \n{fp['J']}")

    return results

def plot_fixed_points_on_phase_plane(ax, fps, marker_sizes=(70, 70)):
    """
    固定点を既存の (r,f) 位相平面に重畳表示する。
    安定: filled circle, 不安定: open circle, 鞍点: cross。
    """
    if fps is None or len(fps) == 0:
        return
    for fp in fps:
        r, f = fp["r"], fp["f"]
        kind = fp["kind"]
        if kind in ("stable node", "stable focus"):
            ax.scatter(r, f, s=marker_sizes[0], c="k", marker="o", zorder=5, label=None)
        elif kind == "saddle" or "saddle" in kind:
            ax.scatter(r, f, s=marker_sizes[1], c="red", marker="x", zorder=5, label=None)
        else:  # unstable node/focus
            ax.scatter(r, f, s=marker_sizes[1], edgecolor="k", facecolors="none", marker="o", zorder=5, label=None)
        ax.annotate(kind.replace(" ", "\n"), (r, f), textcoords="offset points", xytext=(6, 6), fontsize=8)

def per_cycle_stats_last_window(t, f, last_seconds=8.0):
    """
    最後の last_seconds 秒を取り出し、ピーク間を1周期として
    各周期の [平均, 最大, 最小] を返す。
    返り値: dict(means, maxs, mins, npeaks_after_thresh)
    （per_cycle_amplitude と同じピーク検出・しきい値を用いる）
    """
    t0 = max(0.0, t[-1] - last_seconds)
    mask = t >= t0
    tt = t[mask]; ff = f[mask]
    if ff.size < 200:
        return dict(means=[], maxs=[], mins=[], npeaks_after_thresh=0)

    d = np.diff(ff)
    peak_idx = np.where((d[:-1] > 0) & (d[1:] <= 0))[0] + 1
    if peak_idx.size < 4:
        return dict(means=[], maxs=[], mins=[], npeaks_after_thresh=int(peak_idx.size))

    amp_global = float(ff.max() - ff.min())
    thr = float(ff.min() + 0.3 * amp_global)
    peak_idx = peak_idx[ff[peak_idx] >= thr]
    if peak_idx.size < 2:
        return dict(means=[], maxs=[], mins=[], npeaks_after_thresh=int(peak_idx.size))

    means, maxs, mins = [], [], []
    for k in range(peak_idx.size - 1):
        i0, i1 = peak_idx[k], peak_idx[k+1]
        if i1 <= i0 + 3:
            continue
        tseg = tt[i0:i1+1]
        fseg = ff[i0:i1+1]
        dur = float(tseg[-1] - tseg[0])
        if dur <= 1e-9:
            continue
        means.append(float(np.trapz(fseg, tseg) / dur))
        maxs.append(float(np.max(fseg)))
        mins.append(float(np.min(fseg)))
    return dict(means=means, maxs=maxs, mins=mins, npeaks_after_thresh=int(peak_idx.size))

# ===========================
# Main sweep + (optional) plotting + phase plane
# ===========================
# def main():
    # Build FI(mu) LUT once
mu_grid, FI_grid = build_FI_mu_lut(P)

# -------------------------
# (Optional) parameter sweep for oscillation map
# -------------------------
# You can customize these. If you keep w_values empty, we skip the map.
U_values = [0.2]#np.linspace(U_min, U_max, nU)  #[0.2]
w_values = [100]#np.linspace(w_min, w_max, nW) #[100]
# U_values = np.linspace(0, 1, 20)
# w_values = [60]               # or: np.linspace(w_min, w_max, nW)
# U_values = [0.45]
# w_values = np.linspace(30, 200, 20)


if len(U_values) > 0 and len(w_values) > 0:
    nU_local = len(U_values)
    nW_local = len(w_values)
    freq_map = np.full((nU_local, nW_local), np.nan, dtype=float)
    osc_map  = np.zeros((nU_local, nW_local), dtype=bool)

    nU = len(w_values)
    f_mean_med = np.full(nU, np.nan, dtype=float)
    f_max_med  = np.full(nU, np.nan, dtype=float)
    f_min_med  = np.full(nU, np.nan, dtype=float)
    
#     for jW, w in enumerate(w_values):
    for iU, U in enumerate(U_values):
        for jW, w in enumerate(w_values):
        
            print("U", U)
            print("w", w)

            t, s, r, f = simulate_mf_3var_w(P, U=U, w=w, tau_s=TAU_S, K=K,
                                            mu_grid=mu_grid, FI_grid=FI_grid,
                                            Tsim=Tsim, dt=dt_mf)
            _, is_osc, _ = per_cycle_amplitude(t, f, last_seconds=LAST_SECONDS,
                                               min_peaks=MIN_PEAKS, amp_thr=AMP_THR)
            if is_osc:
                freq = estimate_frequency_hz(t, f, last_seconds=LAST_SECONDS)
                freq_map[iU, jW] = freq
                osc_map[iU, jW] = True
            stats = per_cycle_stats_last_window(t, f, last_seconds=LAST_SECONDS)
            if len(stats["means"]) == 0:
                continue

#             f_mean_med[iU] = float(np.median(stats["means"]))
#             f_max_med[iU]  = float(np.median(stats["maxs"]))
#             f_min_med[iU]  = float(np.median(stats["mins"]))
            
            f_mean_med[jW] = float(np.median(stats["means"]))
            f_max_med[jW]  = float(np.median(stats["maxs"]))
            f_min_med[jW]  = float(np.median(stats["mins"]))
            
#             print(f"[saved] {out}")


                
            Plot f (left y-axis) and r (right y-axis) over the last window
            -------------------------
            fig, ax1 = plt.subplots(figsize=(8, 4))

            # 時間窓マスク（最後の LAST_SECONDS 秒）
            t0 = Tsim - LAST_SECONDS
            mask = t >= t0

            # 左軸: f(Hz)
            lns1 = ax1.plot(t[mask], f[mask], color="tab:blue", label="f (Hz)")
            ax1.set_xlabel("Time (sec)")
            ax1.set_ylabel("f (Hz)", color="black")
            # f の y 範囲（安全マージンつき）
            f_seg = f[mask]
#             if f_seg.size > 0:
#                 ax1.set_ylim(0, np.max(f_seg) + 5)
            ax1.set_ylim(0, 65)
            ax1.tick_params(axis="y", labelcolor="black")

            # 右軸: r
            ax2 = ax1.twinx()
            lns2 = ax2.plot(t[mask], r[mask], color="tab:orange", label="r")
            ax2.set_ylabel("r", color="black")
            ax2.set_ylim(0.0, 1.0)
            ax2.tick_params(axis="y", labelcolor="black")

            # 凡例を統合
            lns = lns1 + lns2
            labels = [l.get_label() for l in lns]
#             ax1.legend(lns, labels, loc="upper right")

            plt.xlim(t0, Tsim)
            plt.xlim(8, 12)
            plt.tight_layout()
            plt.savefig(fig_dir + "/mean-field_f-r-plot_K{}_U{}_w{}_longy.SVG".format(K, U, w))
            plt.show()
#             -------------------------
#             # Plot f comparison (last 2s)
#             # -------------------------
#             plt.figure(figsize=(8, 4))
#         #     plt.plot(t_net[mask], f_net[mask], label="LIF net: f̂(t)")
#             plt.plot(t, f, color = "tab:blue")
#             plt.xlabel("Time (sec)"); plt.ylabel("f (Hz)")
#             plt.xlim(Tsim-LAST_SECONDS, Tsim)
#             plt.ylim(0, np.max(f[(Tsim-LAST_SECONDS)*1000: Tsim*1000]) + 5)
# #                 plt.title(f"f(t) comparison | tau_s={ms} ms | w={w_eff:.3f} mV | corr={cf:.3f} RMSE={rmse_f:.2f}")
# #                 plt.grid(True, alpha=0.3); 
# #             plt.legend()
#             plt.show()

#             # -------------------------
#             # Plot r comparison (last 2s)
#             # -------------------------
#             plt.figure(figsize=(8, 4))
#         #     plt.plot(t_net[mask], r_net[mask], label="LIF net: <r(t)>")
#             plt.plot(t, r, color = "tab:orange")
#             plt.xlabel("Time (sec)"); plt.ylabel("r")
#             plt.xlim(Tsim-LAST_SECONDS, Tsim)
#             plt.ylim(0,1)
# #                 plt.title(f"r(t) comparison | tau_s={ms} ms | corr={cr:.3f} RMSE={rmse_r:.4f}")
# #                 plt.grid(True, alpha=0.3); 
#             plt.legend()
#             plt.show()

#             draw_phase_plane_r_f(P, U=U, w=w, tau_s=TAU_S, K=K,
#                      mu_grid=mu_grid, FI_grid=FI_grid,
#                      s_mode="adiabatic",   # change to "fixed" if you prefer
#                      s_fixed=None,
#                      r_lim=(0.0, 1.1),
#                      f_lim=(0.0, 150.0),
#                      n_grid=220,
#                      density=1.2,
#                      figsize=(7.2, 5.6))

      #  Plot: yellow gradient for freq where oscillatory; black where not
#         data = np.ma.masked_invalid(freq_map)

#         yellow_cmap = LinearSegmentedColormap.from_list(
#             "yellow_grad", ["#fffde7", "#fff59d", "#ffeb3b", "#fdd835"]
#         )
#         yellow_cmap.set_bad(color="navy")  # non-oscillatory -> black

#         if np.isfinite(freq_map).any():
#             vmin = 0.0
#             vmax = float(np.nanpercentile(freq_map, 99))
#             if not np.isfinite(vmax) or vmax <= vmin:
#                 vmax = vmin + 1.0
#         else:
#             vmin, vmax = 0.0, 1.0

#         plt.figure(figsize=(8, 6))
#         im = plt.imshow(
#             data,
#             origin="lower",
#             aspect="auto",
#             extent=(w_values[0], w_values[-1], U_values[0], U_values[-1]),
#             cmap=yellow_cmap,
#             vmin=vmin, vmax=vmax,
#             interpolation="nearest"
#         )
#         cbar = plt.colorbar(im)
#         cbar.set_label("Frequency (Hz)  [yellow = oscillatory; black = non-oscillatory]")

#         plt.xlabel("w (mV)")
#         plt.ylabel("U")
#         plt.title(f"Mean-field oscillation map_T15s_navy (tau_s={TAU_S*1e3:.0f} ms, K={K})")
#         plt.tight_layout()
#         plt.savefig(fig_dir + "/mean-field_2D_oscillation_map_T15s.SVG")
#         plt.show()
        
        ###freq map viridis
#         data = np.ma.masked_invalid(freq_map)

#         # Use viridis and set NaN/masked values to the lowest viridis color.
#         viridis_cmap = plt.get_cmap("viridis").copy()
#         viridis_cmap.set_bad(color=viridis_cmap(0.0))

#         if np.isfinite(freq_map).any():
#             vmin = 0.0
#             vmax = float(np.nanpercentile(freq_map, 99))
#             if not np.isfinite(vmax) or vmax <= vmin:
#                 vmax = vmin + 1.0
#         else:
#             vmin, vmax = 0.0, 1.0

#         plt.figure(figsize=(8, 6))
#         im = plt.imshow(
#             data,
#             origin="lower",
#             aspect="auto",
#             extent=(w_values[0], w_values[-1], U_values[0], U_values[-1]),
#             cmap=viridis_cmap,
#             vmin=vmin,
#             vmax=vmax,
#             interpolation="nearest"
#         )

#         cbar = plt.colorbar(im)
#         cbar.set_label("Frequency (Hz)  [viridis; lowest color = non-oscillatory]")

#         plt.xlabel("w (mV)")
#         plt.ylabel("U")
#         plt.title(f"Mean-field oscillation map_T15s_viridis (tau_s={TAU_S*1e3:.0f} ms, K={K})")
#         plt.tight_layout()
#         plt.savefig(fig_dir + "/mean-field_2D_oscillation_map_T15s_viridis.SVG")
#         plt.show()


#    #plot firing rates in different U and w
#     plt.figure(figsize=(7.2, 4.6))
# #     plt.plot(U_values, f_mean_med, color="tab:blue",  marker="o", ms=3, lw=1.8, label="cycle mean f")
# #     plt.plot(U_values, f_max_med,  color="tab:orange", marker="^", ms=3, lw=1.8, label="cycle max f")
# #     plt.plot(U_values, f_min_med,  color="tab:green", marker="v", ms=3, lw=1.8, label="cycle min f")
    
#     plt.plot(w_values, f_mean_med, color="tab:blue",  marker="o", ms=3, lw=1.8, label="cycle mean f")
#     plt.plot(w_values, f_max_med,  color="tab:orange", marker="^", ms=3, lw=1.8, label="cycle max f")
#     plt.plot(w_values, f_min_med,  color="tab:green", marker="v", ms=3, lw=1.8, label="cycle min f")
    
# #     plt.xlabel("U")
#     plt.xlabel("w")
#     plt.ylabel("f (Hz)")
# #             plt.title(f"Cycle-wise f statistics vs U  (w={w:.1f} mV, K={K}, window={last_seconds:.0f}s)")
#     plt.legend(loc="best")
#     # Y 範囲は見やすいように自動調整
#     y_all = np.concatenate([np.nan_to_num(f_mean_med, nan=np.nan),
#                             np.nan_to_num(f_max_med,  nan=np.nan),
#                             np.nan_to_num(f_min_med,  nan=np.nan)])
#     if np.isfinite(y_all).any():
#         ymin = max(0.0, float(np.nanmin(y_all)) - 2.0)
#         ymax = float(np.nanmax(y_all)) + 2.0
#         if ymax <= ymin:  # 保険
#             ymax = ymin + 1.0
#         plt.ylim(ymin, ymax)

#     plt.tight_layout()
# #     plt.savefig(fig_dir + "/mean-field_FR_K{}_changingU_w{}.SVG".format(K,w))
#     plt.savefig(fig_dir + "/mean-field_FR_K{}_changingw_U{}.SVG".format(K,U))
    
#     plt.show()
else:
    print("[sweep] Skipped oscillation map (set non-empty U_values and w_values to enable).")

# -------------------------
# NEW: (r,f) phase plane + nullclines + streamplot
# -------------------------
# Choose the parameter point you want to inspect on the phase plane:
#     U_pp = 0.5
#     w_pp = 60.0  # mV

# s handling:
#   "adiabatic" -> s = tau_s * U * r * f (recommended; fast synapse)
#   "fixed"     -> s is fixed to s_fixed (if None, it is estimated by a short MF run)
#     draw_phase_plane_r_f(P, U=U_pp, w=w_pp, tau_s=TAU_S, K=K,
#                          mu_grid=mu_grid, FI_grid=FI_grid,
#                          s_mode="adiabatic",   # change to "fixed" if you prefer
#                          s_fixed=None,
#                          r_lim=(0.0, 1.1),
#                          f_lim=(0.0, 150.0),
#                          n_grid=220,
#                          density=1.2,
#                          figsize=(7.2, 5.6))

# if __name__ == "__main__":
#     main()